# Calibration Notebook

This notebook carries out the parsimonious calibration workflow:

1. Load the pooled historical mark distribution \((p_1,p_2,p_3)\).
2. Input market observables for a specific game.
3. Infer the two scoring intensities \(\lambda_A\) and \(\lambda_B\) under the restriction that both teams share the same mark distribution.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt

project_root = Path.cwd()
if not (project_root / "src").exists() and (project_root.parent / "src").exists():
    project_root = project_root.parent

src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from Calibration import Calibration, load_distribution

## Load pooled scoring distribution

In [ ]:
distribution_path = project_root / "data" / "processed" / "scoring_distribution_2024-25_regular_season.csv"
distribution, p = load_distribution(distribution_path)
distribution

## Market inputs

In [ ]:
team_a = "Team A"
team_b = "Team B"

# Use minutes for the horizon.
T = 48.0

# Replace these with observed market values for the selected game.
market_yes_prob = 0.58
market_total_points = 228.5

# Monte Carlo controls for the calibration step.
num_simulations = 100_000
seed = 42

In [ ]:
calibration = Calibration.from_distribution_csv(
    distribution_path=distribution_path,
    horizon=T,
    market_yes_prob=market_yes_prob,
    market_total_points=market_total_points,
    num_simulations=num_simulations,
    seed=seed,
)

print(f"p1 = {calibration.p[0]:.6f}")
print(f"p2 = {calibration.p[1]:.6f}")
print(f"p3 = {calibration.p[2]:.6f}")
print(f"Expected points per scoring event = {calibration.expected_points_per_event:.6f}")
print(f"Implied lambda_A + lambda_B = {calibration.lambda_sum:.6f} scoring events per minute")

## Scan the one-dimensional calibration problem

In [ ]:
scan = calibration.run_scan(num_grid=31)
scan

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(scan["lambda_a"], scan["model_yes_prob"], marker="o", linewidth=1.5)
ax.axhline(calibration.market_yes_prob, color="red", linestyle="--", linewidth=1.5, label="Market yes probability")
ax.set_title("Model Yes Probability as a Function of $\\lambda_A$")
ax.set_xlabel("$\\lambda_A$")
ax.set_ylabel("Model Team A win probability")
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## Solve for the implied intensities

In [ ]:
calibration_results = calibration.calibrate(num_grid=31)
calibration_results

In [ ]:
calibration.summary(team_a=team_a, team_b=team_b)